# Gaze-driven VLM input on Phi-3.5-vision: Full gate (when) + foveated tokens (where)

The same pipeline as the Qwen2.5-VL, Gemma-3, InternVL3.5 and Qwen3-VL notebooks, on
**Phi-3.5-vision-instruct** (CLIP ViT-L/14-336 + Phi-3-mini). Phi's image input is its "HD"
layout: with the processor's default `num_crops = 4` a frame becomes a 672 px view in 2 x 2 crops of
336 px (576 tokens, a row-break token after each of the 24 rows), a separator, and the whole frame at
336 px (the global view, 156 tokens): **757 tokens per frame**.

```
gaze + IMU -> Full gate (eye + head model) -> send policy (POLICY, default D+P): which frames are sent
sent frame -> Phi's 672 px view + gaze -> foveated cells BEFORE CLIP: 28 px cells at the gaze,
     56 / 112 px cells shrunk to 28 px further out (never across a 336 px crop); each cell = 2 x 2
     patches of 14 px with CLIP's position table read at the cell's quadrant centres
  -> CLIP per crop (its own weights) -> Phi's 2 x 2 merge, row breaks, separator, global view
  -> Phi's projection -> Phi-3-mini                                        (no training)
```

Foveation thins out the high-resolution crop tokens; **Phi's own global view is kept in every
condition** (it is the model's built-in low-resolution periphery). The ring settings are the other
models': on Phi's 672 px view they keep the same share of high-resolution tokens.

| condition (all on the gate's frames) | crop tokens | tokens per frame |
|---|---|---|
| Phi stock (num_crops = 4) | 576 | 757 |
| **fovea at gaze**, 4 ring settings | ~113 / 233 / 333 / 419 | ~290 / 410 / 510 / 600 |
| uniform: each crop downscaled, closest crop tokens | ~100 / 256 / 324 / 400 | similar |
| fovea at image centre, third setting | ~333 | ~510 |

(Token counts are estimates; section 6 measures them on the sent frames.) The model is not gated.
**No remote code is needed to run it**: CLIP, Phi's separator tokens and projection, and the LLM (as
transformers' native `Phi3ForCausalLM`, SDPA attention) are loaded from the checkpoint. Phi's remote
code is loaded once, in section 6, as the parity reference. Scoring takes ~4-5 h on an A100
(stock prompts are ~36k tokens); it is resumable.

## Run order

1. Clone, mount, install, model check  2. AEA links  3. Questions  4. Full gate + send policy
5. Frames and gaze  6. Phi: unit tests, parity with Phi's own pipeline  7. Score  8. Results

## 1. Clone, mount, install, model check

In [ ]:
import os, sys, json, subprocess

REPO, BRANCH = '/content/GazeVLM', 'eyehead-gate'     # 'main' once merged
URL = 'https://github.com/shubhamOjha1000/GazeVLM-HWSW-Codesign.git'
if os.path.exists(os.path.join(REPO, '.git')):
    subprocess.run(['git', '-C', REPO, 'fetch', '-q', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO, 'checkout', '-q', '-B', BRANCH, f'origin/{BRANCH}'], check=True)
else:
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, URL, REPO], check=True)
os.chdir(REPO)
sys.path.insert(0, REPO)
print(subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)
assert os.path.exists('vlmeval/phi_fovea.py'), f'vlmeval/phi_fovea.py is not on branch {BRANCH!r}'

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub', 'pytest'], check=True)

import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/GazeVLM'
FOLDS     = os.path.join(DRIVE_DIR, 'folds')
RUNS      = os.path.join(DRIVE_DIR, 'runs')
VLM_DIR   = os.path.join(DRIVE_DIR, 'vlm', 'egoeverything')
FRAMES_TAR = os.path.join(VLM_DIR, 'frames_tar')          # one .tar per video and size, reused later
LOCAL_HI = '/content/frames_1344'
os.makedirs(VLM_DIR, exist_ok=True)

CKPT_FULL = os.path.join(RUNS, 'eyehead')                  # the Full model's 5 fold checkpoints
missing = [i for i in range(1, 6) if not os.path.exists(os.path.join(CKPT_FULL, f'fold{i}.pt'))]
assert not missing, f'missing fold checkpoints {missing} in {CKPT_FULL} -- run the eyehead notebook first'
print('Full model checkpoints: 5 folds found in', CKPT_FULL)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

In [ ]:
# Phi-3.5-vision-instruct (remote code on the hub; this pipeline runs it on native transformers parts)
from huggingface_hub import login, model_info
from vlmeval import phi_fovea as ph
try:
    from google.colab import userdata
    _tok = userdata.get('HF_TOKEN')
    if _tok:
        login(token=_tok)
except Exception:
    pass
MODEL_ID = ph.MODEL_ID
model_info(MODEL_ID)
import transformers
print('model:', MODEL_ID, '| transformers', transformers.__version__)
assert hasattr(transformers, 'Phi3ForCausalLM'), 'this transformers has no Phi3ForCausalLM: pip install -U transformers'

## 2. The AEA download-links JSON

The signed links expire after ~14 days. This cell checks the copy on Drive:

- **missing or expired** → it asks you to upload the fresh `aea_download_urls.json`, checks
  its links work, and **replaces the older copies** on Drive (`MyDrive/GazeVLM/` and, if
  present, `MyDrive/aea/`, so every notebook picks up the fresh one)
- **still valid** → nothing to do

Set `FORCE_UPLOAD = True` to replace it anyway.

In [ ]:
from vlmeval.frames import links_fresh, install_links

FORCE_UPLOAD = False
LINKS = os.path.join(DRIVE_DIR, 'aea_download_urls.json')
OLD_COPIES = [LINKS] + [p for p in ['/content/drive/MyDrive/aea/aea_download_urls.json']
                        if os.path.exists(p)]

ok, msg = links_fresh(LINKS) if os.path.exists(LINKS) else (False, 'not on Drive')
print('Drive copy:', msg)
if FORCE_UPLOAD or not ok:
    from google.colab import files
    print('upload the fresh aea_download_urls.json ...')
    up = files.upload()
    name = list(up.keys())[0]
    ok, msg = links_fresh(name)
    print('uploaded file:', msg)
    assert ok, 'the uploaded JSON does not work either -- download a fresh one from projectaria.com/datasets/aea'
    install_links(name, OLD_COPIES)
    os.remove(name)                      # the copies live on Drive; keep the repo clean
    for p in OLD_COPIES:
        print('   replaced', p)
links = json.load(open(LINKS))
print(f"using {LINKS}: {len(links['sequences'])} sequences")
print('(the eyehead notebook\'s IMU calibration, section 3, can now download its VRS too)')

## 3. EgoEverything questions on the gate's videos

In [ ]:
from vlmeval import egoev

qs_all = egoev.load_questions()
aea = qs_all[qs_all.video.str.startswith('loc')]
clips_all, test = egoev.gate_clips(FOLDS, test_only=False)
gate_videos = {c['sequence'] for c in clips_all}
test_videos = {c['sequence'] for c in clips_all if c['is_test']}

# DEMO = True: only the 4 locked TEST videos (204 questions), to check that everything works.
# DEMO = False: all 12 gate videos with EgoEverything questions (834 questions).
DEMO = False
qs = aea[aea.video.isin(gate_videos)].reset_index(drop=True)
if DEMO:
    qs = qs[qs.video.isin(test_videos)].reset_index(drop=True)
clips = [c for c in clips_all if c['sequence'] in set(qs.video)]
videos = [c['sequence'] for c in clips]
n_frames = {c['sequence']: len(c['frame_no']) for c in clips}
print(f'EgoEverything: {len(qs_all):,} questions, {len(aea):,} on AEA ({aea.video.nunique()} videos)')
print(f'on the gate videos: {len(qs):,} questions on {len(videos)} videos '
      f'({qs.video.isin(test_videos).sum()} questions on {len(set(videos) & test_videos)} test videos)\n')

per = qs.groupby('video').agg(questions=('qid', 'size'), t0_max=('t0', 'max'),
                              median_dist2gaze=('dist2gaze', 'median'))
per['video_s'] = [n_frames[v] for v in per.index]
per['split'] = ['test' if v in test_videos else 'train (held out by its fold)' for v in per.index]
display(per.sort_values('questions', ascending=False).round(1))
print('answer letters:', qs.answer.value_counts().sort_index().to_dict())

## 4. Full gate + policy D: which frames are sent

Each training video is scored by the fold model that held it out; test videos by the 5-fold
ensemble (cached on Drive). The held-out 4-class accuracy should be close to the
cross-validation numbers (~0.6), not higher, which would mean a video leaked into its own
training. Policy D then sends a frame when the view settles after motion, and on each
REFIXATION.

In [ ]:
from eyehead.oof import oof_predictions

OOF = os.path.join(VLM_DIR, 'oof_preds_full.csv')
if os.path.exists(OOF):
    oof = pd.read_csv(OOF)
    print('loaded', OOF)
else:
    oof = oof_predictions(FOLDS, CKPT_FULL)
    oof.to_csv(OOF, index=False)
    print('wrote', OOF)

o12 = oof[oof.sequence.isin(videos)]
print(f"held-out 4-class accuracy on these {len(videos)} videos: {(o12.pred == o12.quad).mean():.3f}")

labels = {'full': egoev.video_labels(clips, oof)}
# Temporal policy: 'D' = send when the view settles + on each REFIXATION;
# 'D+P' = D + the first frame of every PURSUIT run; 'D+P all' = D + every PURSUIT frame.
POLICY = 'D+P'
POLICY_TAG = {'D': 'D', 'D+P': 'DP', 'D+P all': 'DPall'}[POLICY]   # separate result files per policy
sel = {'D: full': egoev.select_conditions(clips, labels, main='full')[f'{POLICY}: full']}
D = sel['D: full']
per = pd.DataFrame({'video_s': n_frames, 'frames sent': {v: len(D[v]) for v in videos}})
per['one frame every (s)'] = (per.video_s / per['frames sent']).round(1)
display(per)
print(f"policy {POLICY} sends {60 * per['frames sent'].sum() / per.video_s.sum():.1f} frames/min "
      f"({per['frames sent'].mean():.1f} per video)")

## 5. Frames and gaze

The 1344 px frames cached on Drive by the Qwen notebook are reused; each is resized exactly as
Phi's processor resizes it (the 672 px view and the 336 px global view); videos not cached yet are
downloaded once. One gaze point per sent frame: the densest fixation cluster in the second before
it (`vlmeval.spatial.gaze_fixation`).

In [ ]:
from vlmeval.frames import ensure_frames, frame_path
from vlmeval import qwen_fovea as qf, spatial as sp

metas = {}
for v in videos:
    metas[v] = ensure_frames(v, links, LOCAL_HI, FRAMES_TAR, size=qf.IMG)
    m = metas[v]
    flag = 'OK' if m['n'] >= n_frames[v] else f'!! only {m["n"]} frames, labels need {n_frames[v]}'
    print(f"   {v:<26} {m['n']:4d} frames  {flag}")

def frame_time(v, k):
    return metas[v]['ts'][k] if k < metas[v]['n'] else k

tracks, n_drop = sp.gaze_tracks(FOLDS, videos, frame_time)
frames_of = {}
for v in videos:
    m = metas[v]
    frames_of[v] = []
    for f in D[v]:
        if f >= m['n']:
            continue
        t = float(m['ts'][f])
        gz_ = sp.gaze_fixation(tracks[v], t) if v in tracks else None
        frames_of[v].append(dict(key=(v, int(f)), path=frame_path(m, f), t=t,
                                 gaze=tuple(map(float, gz_)) if gz_ is not None else None))
items = [dict(qid=int(q.qid), video=q.video, question=q.question, options=list(q.options),
              answer=q.answer, frames=frames_of[q.video])
         for q in qs.sort_values(['video', 'qid']).itertuples()]
nf = np.array([len(it['frames']) for it in items])
print(f"{len(items)} questions on {len(videos)} videos; frames per question: mean {nf.mean():.1f}, max {nf.max()}; "
      f"gaze missing on {sum(fr['gaze'] is None for v in videos for fr in frames_of[v])} of "
      f"{sum(len(frames_of[v]) for v in videos)} sent frames")

## 6. Phi-3.5-vision: unit tests and parity with Phi's own pipeline

| check | pass |
|---|---|
| unit tests (CPU, a tiny random CLIP from transformers' own `CLIPVisionModel` on the real 24 x 24 grid): layouts never cross a crop, **HD assembly = Phi's own reshape / row-break code**, checkpoint names load onto this CLIP, position table exact at grid points, **all-28 px layout through our cells path = CLIP's own forward on the 4 crops**, tokens follow their cells, uniform crops keep Phi's structure, prompt tags and **Phi's chunked tokenization (a BOS per chunk)**, answer-letter tokens; plus the Qwen2.5-VL tests (shared helpers) | all pass |
| real model, one frame, **vision in fp32**: our cells path vs CLIP's own forward on Phi's crops | cosine > 0.9999 |
| Phi's remote code, where this transformers can still load it: its processor's **pixels and token ids** vs ours; its model (eager attention) vs our native path: image embeddings, answer probabilities, and the LLMs alone | pixels equal, ids equal, same answer |
| feature drift: cosine of foveated crop tokens with Phi's own crop tokens over the same area | 28 px cells close to 1 |

In [ ]:
r = subprocess.run([sys.executable, '-m', 'pytest', 'tests/test_phi_fovea.py', 'tests/test_qwen_fovea.py',
                    'tests/test_gaze_fixation.py', '-q', '-rs'], capture_output=True, text=True, cwd=REPO)
import transformers
print('transformers', transformers.__version__, '| torch', torch.__version__)
print(r.stdout[-4000:], r.stderr[-2000:])
assert r.returncode == 0, 'unit tests failed: fix before using the GPU'

from vlmeval import qwen_mcq, fovea_registry as reg      # same prompt (frame times, question, options)
img_m, llm, tok, letter_ids = ph.load_model(MODEL_ID)
print(MODEL_ID, f'{torch.cuda.memory_allocated()/1e9:.1f} GB on GPU | answer tokens',
      tok.convert_ids_to_tokens(letter_ids))

q0 = qs.iloc[0]
m0 = metas[q0.video]
k0 = int(np.argmin(np.abs(np.array(m0['ts']) - q0.t0)))
pil0 = ph.load_pil(frame_path(m0, k0))
CHECK_REMOTE = True        # load Phi's own remote-code model once as the parity reference (~8 GB, a few min)
remote, remote_proc, notes = ph.load_remote(MODEL_ID) if CHECK_REMOTE else (None, None, {})
rep = ph.parity_report(img_m, llm, tok, letter_ids, pil0, q0.question, q0.options, qwen_mcq.build_messages,
                       remote, remote_proc)
print("parity, all-28 px layout through our path vs Phi's own computation:")
for k_, v_ in {**notes, **rep}.items():
    print(f'   {k_:<42} {v_}')
del remote
torch.cuda.empty_cache()

num = lambda k: isinstance(rep.get(k), float)
ok = rep['vision_fp32_cos_min'] > 0.9999 and rep['tokens_per_frame'] == 757
if num('remote_pixel_max_abs_diff'):
    ok &= rep['remote_pixel_max_abs_diff'] < 1e-3
if 'remote_token_ids_equal' in rep:
    ok &= rep['remote_token_ids_equal']
if rep.get('remote_clip_weights_equal') and num('remote_probs_max_abs_diff'):
    ok &= rep['remote_image_embeds_cos_min'] > 0.99 and rep['remote_same_answer']
elif CHECK_REMOTE:
    print("!! Phi's remote model could not serve as the reference in this transformers version (see notes): "
          "only the vision path and, where it loaded, the processor were checked against Phi")
assert ok, 'parity failed: do not trust the foveated results'
print('parity OK')

# ring settings: the other models' (R, a); on the 672 px view they keep the same share of crop tokens
PICKS = reg.RINGS
gz_D = [fr['gaze'] or (0.5, 0.5) for v in videos for fr in frames_of[v]]
def budget_of(p):
    lay = [ph.foveated_cells(*g_, a=p['a'], R_deg=p['R']) for g_ in gz_D]
    crop = np.mean([len(c) for c in lay])
    rows = np.mean([len(np.unique(c[:, 0] // ph.BASE)) for c in lay])
    return {'R': p['R'], 'a': p['a'], 'crop tokens': crop, 'tokens per frame': crop + rows + 1 + 156,
            'uniform crop tokens': 4 * ph.same_tokens_k(crop) ** 2}
budget = pd.DataFrame({p['label']: budget_of(p) for p in PICKS}).T
budget['saving vs 757'] = (100 * (1 - budget['tokens per frame'] / 757)).round(1).astype(str) + '%'
print('\ntoken budgets (Phi stock = 576 crop tokens, 757 per frame; mean over the sent frames):')
display(budget.round(1))

g0 = sp.gaze_fixation(tracks[q0.video], m0['ts'][k0]) or (0.5, 0.5)
p_mid = PICKS[len(PICKS) // 2]
hd0, _ = ph.views(pil0)
cells0 = ph.foveated_cells(*g0, a=p_mid['a'], R_deg=p_mid['R'])
print(f"feature drift on one frame ({len(cells0)} crop tokens, R={p_mid['R']}, a={p_mid['a']}):")
display(ph.fovea_feature_drift(img_m, hd0, cells0).round(3))

COLS = {28: 'white', 56: 'yellow', 112: 'deepskyblue'}
v_ = videos[0]
fr_ = frames_of[v_][len(frames_of[v_]) // 2]
im_ = ph.views(ph.load_pil(fr_['path']))[0].permute(1, 2, 0).numpy()
g_ = fr_['gaze'] or (0.5, 0.5)
fig, axs = plt.subplots(1, len(PICKS) + 1, figsize=(3.6 * (len(PICKS) + 1), 3.9))
axs[0].imshow(im_); axs[0].set_title('Phi stock: 576 crop tokens', fontsize=8); axs[0].axis('off')
for a_, p in zip(axs[1:], PICKS):
    cells = ph.foveated_cells(*g_, a=p['a'], R_deg=p['R'])
    a_.imshow(im_)
    for y, x, s in cells:
        a_.add_patch(mpatches.Rectangle((x, y), s, s, fill=False, ec=COLS[s], lw=0.6))
    a_.axhline(ph.CROP, color='red', lw=0.8); a_.axvline(ph.CROP, color='red', lw=0.8)
    a_.plot(g_[0] * ph.HD, g_[1] * ph.HD, 'r+', ms=12, mew=2)
    a_.set_title(f"{len(cells)} crop tokens (R={p['R']}, a={p['a']})", fontsize=8); a_.axis('off')
plt.suptitle("one crop token per box: white 28 px (= Phi's own), yellow 56 px, blue 112 px; red lines: the 2 x 2 "
             "crops; + = gaze. Phi's global view (156 tokens) is in every condition", fontsize=9)
plt.tight_layout(); plt.show()

## 7. Score: the gate's frames, Phi stock vs foveated vs uniform crops

Every question gets all the frames the send policy sent on its video. Work runs video by video
and, within a video, condition by condition: each frame's views and global-view tokens are made
once per video, its condition tokens once per (video, condition). Answer = argmax of the raw
next-token logits over A-E (Phi writes the letter right after `<|assistant|>`). Resumable.

In [ ]:
fov, UNI, STOCK = reg.phi_fovea, reg.phi_uniform, reg.PHI_STOCK
CONDS = {STOCK: ('stock', None)}
for p in PICKS:
    CONDS[fov(p)] = ('fovea', dict(a=p['a'], R=p['R'], gaze='gaze'))
    CONDS[UNI(p)] = ('uniform_match', fov(p))
CONDS[fov(p_mid, 'centre')] = ('fovea', dict(a=p_mid['a'], R=p_mid['R'], gaze='centre'))
print(f'{len(items)} questions x {len(CONDS)} conditions:', *CONDS, sep='\n   ')

OUT = os.path.join(VLM_DIR, f"{MODEL_ID.split('/')[-1]}_{POLICY_TAG}_fovea.jsonl")
ph.run_questions(img_m, llm, tok, letter_ids, items, CONDS, OUT, qwen_mcq.build_messages)

## 8. Results

- accuracy per condition (95% interval from resampling videos), tokens per frame, saving vs 757
- paired tests at each ring setting: fovea vs uniform crops, fovea vs stock
- **one pre-specified test across budgets**: per question, the mean over the 4 fovea settings minus
  the mean over the 4 matched uniform settings
- accuracy vs tokens plot, and **all VLMs side by side** (same frames; budget levels 1-4 = the
  four ring settings, or SmolVLM's four k; a model shows up once its notebook has run with the
  same POLICY)

In [ ]:
res = egoev.load_results(OUT)
res = res[res.video.isin(videos) & res.condition.isin(CONDS)]
bad = res[res['error'].notna()] if 'error' in res else res.iloc[:0]
if len(bad):
    print(f'{len(bad)} rows not scored, left out:'); display(bad.groupby(['condition', 'error']).size())
res = res[res['error'].isna()].copy()
res['correct'] = res['correct'].astype(int)

acc = egoev.accuracy_table(res)
acc['tokens / frame'] = res.groupby('condition').n_image_tokens.mean() / acc['frames_per_q']
acc['crop tokens / frame'] = res.groupby('condition').n_crop_tokens.mean() / acc['frames_per_q']
acc['saving vs 757'] = 100 * (1 - acc['tokens / frame'] / 757)
acc = acc.loc[[c for c in CONDS if c in acc.index]]
print(f'Phi-3.5-vision, {res.qid.nunique()} questions on {res.video.nunique()} videos, policy {POLICY} frames:')
display(acc[['n', 'accuracy', 'ci_low', 'ci_high', 'tokens / frame', 'crop tokens / frame', 'saving vs 757']].round(1))

rows = []
for p in PICKS:
    for a_, b_ in ((fov(p), UNI(p)), (fov(p), STOCK)):
        if {a_, b_} <= set(res.condition):
            rows.append(dict(budget=p['label'], **egoev.paired_test(res, a_, b_)))
if {fov(p_mid), fov(p_mid, 'centre')} <= set(res.condition):
    rows.append(dict(budget=p_mid['label'], **egoev.paired_test(res, fov(p_mid), fov(p_mid, 'centre'))))
pt = pd.DataFrame(rows)
print('paired on the same questions (diff = a - b, accuracy points; CI from resampling videos):')
display(pt[['budget', 'a', 'b', 'diff', 'ci_low', 'ci_high', 'a_only_right', 'b_only_right', 'p_mcnemar']].round(3))

w = res.pivot_table(index=['qid', 'video'], columns='condition', values='correct')
f_cols, u_cols = [fov(p) for p in PICKS], [UNI(p) for p in PICKS]
if set(f_cols + u_cols) <= set(w.columns):
    d = pd.DataFrame({'video': w.index.get_level_values('video'),
                      'diff': 100 * (w[f_cols].mean(1) - w[u_cols].mean(1)).to_numpy()})
    lo, hi = egoev.video_bootstrap(d, lambda t: t['diff'].mean())
    pos, neg = int((d['diff'] > 0).sum()), int((d['diff'] < 0).sum())
    try:
        from scipy.stats import binomtest
        p_sign = binomtest(pos, pos + neg, 0.5).pvalue
    except ImportError:
        p_sign = float('nan')
    print(f"\nACROSS BUDGETS: fovea - uniform = {d['diff'].mean():+.2f} points, 95% CI [{lo:+.2f}, {hi:+.2f}] "
          f"(videos resampled); questions better / worse with fovea: {pos} / {neg}, sign test p = {p_sign:.3f}")

fig, ax = plt.subplots(figsize=(7, 4.4))
def curve(names, style, label):
    d_ = acc.loc[[n for n in names if n in acc.index]].sort_values('tokens / frame')
    ax.errorbar(d_['tokens / frame'], d_.accuracy, yerr=[d_.accuracy - d_.ci_low, d_.ci_high - d_.accuracy],
                fmt=style, capsize=3, label=label)
curve(f_cols, 'o-', 'Phi: fovea at gaze')
curve(u_cols, 's--', 'Phi: uniform crops, closest tokens')
curve([STOCK], 'k*', 'Phi stock (757)')
curve([fov(p_mid, 'centre')], 'x', 'Phi: fovea at image centre')
ax.set(xlabel='visual tokens per frame (crop tokens + row breaks + separator + 156 global)', ylabel='accuracy (%)',
       title=f'Phi-3.5-vision, policy {POLICY} frames, {res.qid.nunique()} questions (95% CI over videos)')
ax.grid(alpha=.3); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

acc_all, tok_all = reg.side_by_side(VLM_DIR, POLICY_TAG, videos)
print(f"\nall VLMs run so far, policy {POLICY} frames (accuracy %; budget levels 1-4 smallest first):")
display(acc_all.round(1))
print('visual tokens per frame:'); display(tok_all.round(0))

---

## Reading it

| look at | meaning |
|---|---|
| section 6 parity | must pass, or nothing after it is trustworthy |
| **across-budgets test** | the single main result for this model |
| fovea vs stock 757 | the token saving: how few crop tokens keep Phi's accuracy |
| fovea at gaze vs centre | is it the gaze, or the centre bias of head-mounted video? |
| all-VLM table | does the effect hold across model families and input designs (Phi: HD crops + global view)? |

## Caveats

- Training-free: CLIP never saw shrunk cells next to full-resolution ones, nor fractional positions;
  section 6's drift table shows how far the features move.
- Phi's row-break tokens follow the token rows that remain (one per 28 px row that has a cell
  starting in it): a foveated grid has fewer, uneven rows; Phi never saw that either.
- The 156-token global view stays in every condition, so Phi's saving is bounded by it (at most
  ~79% of 757); the matched uniform baseline keeps it too.
- The LLM runs as transformers' `Phi3ForCausalLM` (SDPA) on Phi's checkpoint weights, not through
  the remote code (eager attention); section 6 compares the two where the remote code loads.